# 03: Sparse UOT and morphology-guided correspondence paths

Read measured anchors and morphology features, compute cross-section transport couplings and morphology-guided paths, and save inspectable pairing artifacts.

## 1. Read the input data

Edit the real input paths and method parameters below. Run the cells in order using the `spagc` kernel. Each notebook reads its own inputs; it does not execute another notebook.

In [ ]:
from __future__ import annotations

import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import scipy.sparse as sp
import torch
from IPython.display import display

ROOT = Path.cwd().resolve()
while not (ROOT / "notebooks/03_3d").is_dir():
    if ROOT == ROOT.parent:
        raise FileNotFoundError("Start Jupyter inside the spaGC repository.")
    ROOT = ROOT.parent


if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
RUN_ID = os.environ.get("SPAGC_RUN_ID", "00029_native_v2")
OUT = ROOT / "outputs/03_3d" / RUN_ID
SEED = 29
MODE = "path"
USE_CELLTYPE = True
LABEL_KEY = "cell_class"
SPATIAL_KEY = "spatial_st_corrected"
Z_KEY = "z_um"
FRAME = "00029__g0_registered"
if not RUN_ID or Path(RUN_ID).name != RUN_ID or RUN_ID in {".", ".."}:
    raise ValueError("SPAGC_RUN_ID must be a safe directory name")
DATA = ROOT / "data/3d/00029_g0_native"
ANCHORS = DATA / "st/anchors_celltype_domain_niche"
CELLTYPE_LABELS = DATA / "cell_type_labels.csv"
ARTIFACTS = ROOT / "outputs/03_3d" / RUN_ID
FEATURES = ARTIFACTS / "uni2_features.h5"
PATH_CACHE = ARTIFACTS / "morphology_paths.h5"
if MODE not in ("baseline", "uot", "path"):
    raise ValueError("Unsupported run mode or model mode.")
print({"model_mode": MODE, "output": str(OUT)})
SECTION_RANGE = None
ROI_UM = None
MAX_CELLS_PER_ANCHOR = None
TOP_K = 64
HISTOLOGY_WEIGHT = 1.0
PATH_QC_PAIRS = 8
NUCLEUS_FEATURES = ARTIFACTS / "aligned_nucleus_features.h5"
USE_NUCLEUS_PATH = False
NUCLEUS_WEIGHT = 1.0
if USE_NUCLEUS_PATH and MODE != "path":
    raise ValueError(
        "The optional supplied-nucleus experiment requires full mode and morphology paths."
    )
np.random.seed(SEED)
torch.manual_seed(SEED)

In [ ]:
from spagc._io import require_new
from spagc.reconstruction.features import FeatureStore
from spagc.reconstruction.inputs import load_anchors
from spagc.reconstruction.paths import MorphologyPathCache
from spagc.reconstruction.uot import (
    compute_uot_coupling,
    load_sparse_coupling,
    save_sparse_coupling,
    summarize_sparse_coupling,
)

Read measured endpoint sections and the morphology store produced by feature extraction. Baseline mode intentionally omits morphology.

Read measured anchor sections, retain their expression and physical coordinates, and apply only the explicitly selected spatial or cell subset.

In [ ]:
anchors = load_anchors(
    ANCHORS,
    require_labels=USE_CELLTYPE,
    labels_path=CELLTYPE_LABELS,
    section_range=SECTION_RANGE,
    roi_um=ROI_UM,
    max_cells_per_anchor=MAX_CELLS_PER_ANCHOR,
    seed=SEED,
    label_key=LABEL_KEY,
    spatial_key=SPATIAL_KEY,
    z_key=Z_KEY,
)
store = FeatureStore(FEATURES) if MODE != "baseline" else None

## 2. Prepare spatial, molecular and cell-type inputs

Normalize XY globally for the inherited transport cost while retaining physical XY for candidate selection and morphology queries. Keep the same cell-type vocabulary across anchors.

In [ ]:
all_xy = np.vstack([a.obsm[SPATIAL_KEY] for a in anchors])
minimum = all_xy.min(0)
scale = np.ptp(all_xy, axis=0) + 1e-08
categories = (
    sorted(set(v for a in anchors for v in a.obs[LABEL_KEY].astype(str)))
    if USE_CELLTYPE
    else ["unknown"]
)

## 3. Compute transport couplings and morphology-guided paths

Solve candidate-restricted unbalanced transport for each adjacent anchor pair. Save inspectable couplings and evaluate sampled morphology paths through intermediate H&E sections; expression and cell types do not enter the path objective.

In [ ]:
summaries = []
cached_paths = []
for a, b in zip(anchors[:-1], anchors[1:]):
    (xy0, xy1) = (a.obsm[SPATIAL_KEY], b.obsm[SPATIAL_KEY])
    (x0, x1) = ((xy0 - minimum) / scale, (xy1 - minimum) / scale)
    g0 = a.X.toarray() if sp.issparse(a.X) else np.asarray(a.X)
    g1 = b.X.toarray() if sp.issparse(b.X) else np.asarray(b.X)

    def encode_types(obj):
        codes = pd.Categorical(obj.obs[LABEL_KEY], categories=categories).codes
        return np.eye(len(categories), dtype=np.float32)[codes]

    c0 = encode_types(a) if USE_CELLTYPE else np.zeros((a.n_obs, 1), np.float32)
    c1 = encode_types(b) if USE_CELLTYPE else np.zeros((b.n_obs, 1), np.float32)
    (s0, s1) = (str(a.obs.section_id.iloc[0]), str(b.obs.section_id.iloc[0]))
    kwargs = {}
    if store is not None:
        (h0, v0) = store.get_feature(s0, xy0, return_valid=True)
        (h1, v1) = store.get_feature(s1, xy1, return_valid=True)
        kwargs.update(
            h0=h0.numpy(),
            h1=h1.numpy(),
            h0_valid=v0.numpy(),
            h1_valid=v1.numpy(),
            histology_weight=HISTOLOGY_WEIGHT,
        )
    coupling = compute_uot_coupling(
        x0,
        g0,
        c0,
        x1,
        g1,
        c1,
        solver="sparse_topk",
        top_k=min(TOP_K, a.n_obs, b.n_obs),
        candidate_x0=xy0,
        candidate_x1=xy1,
        bidirectional=True,
        use_celltype=USE_CELLTYPE,
        **kwargs,
    )
    assert coupling.total_mass > 0 and np.isfinite(coupling.mass).all()
    cache_path = require_new(OUT / "uot" / f"pair-{s0}-{s1}.npz")
    metadata = {
        "frame": FRAME,
        "source": s0,
        "target": s1,
        "top_k": TOP_K,
        "histology_weight": HISTOLOGY_WEIGHT if store else 0,
        "use_celltype": USE_CELLTYPE,
        "feature_revision": store.revision if store else None,
        "spatial_metric": "legacy_normalized",
        "alpha_spatial": 0.5,
        "uot_reg": 0.8,
        "uot_tau": 0.05,
        "class_weight": 10.0,
    }
    save_sparse_coupling(cache_path, coupling, metadata)
    (restored, restored_metadata) = load_sparse_coupling(cache_path)
    np.testing.assert_array_equal(restored.mass, coupling.mass)
    summaries.append(
        {"pair": f"{s0}-{s1}", **summarize_sparse_coupling(coupling, xy0, xy1)}
    )
    if MODE == "path":
        path_file = PATH_CACHE
        cache = MorphologyPathCache(
            path_file,
            store,
            candidate_radius_um=100,
            candidate_spacing_um=25,
            nucleus_store=FeatureStore(NUCLEUS_FEATURES) if USE_NUCLEUS_PATH else None,
            nucleus_weight=NUCLEUS_WEIGHT if USE_NUCLEUS_PATH else 0.0,
            allow_linear_fallback=False,
        )
        (i, j) = coupling.sample(PATH_QC_PAIRS, rng=np.random.default_rng(SEED))
        keys = [f"{s0}:{a.obs_names[u]}->{s1}:{b.obs_names[v]}" for (u, v) in zip(i, j)]
        ids = cache.prepare(s0, s1, xy0[i], xy1[j], keys)
        (positions, velocities) = cache.evaluate(ids, torch.full((len(ids),), 0.5))
        assert torch.isfinite(positions).all() and torch.isfinite(velocities).all()
        cached_paths.extend(ids)

## 4. Save and inspect transport diagnostics

Save the per-gap transport summary. The path cache retains selected knots, feature revisions and coverage needed by downstream training.

In [ ]:
pd.DataFrame(summaries).to_csv(require_new(OUT / "uot_qc.csv"), index=False)
display(pd.DataFrame(summaries))